In [18]:
## load libraries
from keras.datasets import mnist
import keras
from keras.layers import *
from keras.layers import LeakyReLU
from keras.models import Sequential , Model
from keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt


In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
(X_train , _),(_,_) = mnist.load_data()


11490434/11490434 [==============================] - 0s 0us/step


In [5]:
X_train.shape

(60000, 28, 28)

In [7]:
X_train = X_train-127.5/127.5
print(X_train.min())
print(X_train.max())

-2.0
253.0


In [9]:
total_epochs = 50
batch_size = 256
half_batch = int(batch_size/2)
no_of_batches = int(X_train.shape[0]/batch_size)
noise_dim = 100
adam = Adam(learning_rate= 0.0002, beta_1=0.5)


In [19]:
# Geneator Model : UpSampling
generator = Sequential()
generator.add(Dense(units = 7*7*128, input_shape = (noise_dim,)))
generator.add(Reshape((7,7,128)))
generator.add(LeakyReLU(0.2))
generator.add(BatchNormalization())


In [20]:
#(7,7,128) --> (14,14,64)

generator.add(Conv2DTranspose(filters = 64, kernel_size = (3,3) , strides  = (2,2), padding = 'same'))
generator.add(LeakyReLU(0.2))
generator.add(BatchNormalization())

#(14,14,64) --> (28,28,1)
#(7,7,128) --> (14,14,64)

generator.add(Conv2DTranspose(filters = 1, kernel_size = (4,4) , strides  = (2,2), padding = 'same',activation = 'tanh'))
generator.compile(loss = keras.losses.binary_crossentropy, optimizer = adam)
generator.summary()


#(14,14,64) --> (28,28,1)




Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_5 (Dense)             (None, 6272)              633472    
                                                                 
 reshape_4 (Reshape)         (None, 7, 7, 128)         0         
                                                                 
 leaky_re_lu_5 (LeakyReLU)   (None, 7, 7, 128)         0         
                                                                 
 batch_normalization_2 (Bat  (None, 7, 7, 128)         512       
 chNormalization)                                                
                                                                 
 conv2d_transpose_2 (Conv2D  (None, 14, 14, 64)        73792     
 Transpose)                                                      
                                                                 
 leaky_re_lu_6 (LeakyReLU)   (None, 14, 14, 64)       

In [29]:
# Discriminator Model : Down Sampling
# (28,28,1) -> (14,14,64)

discriminator = Sequential()
discriminator.add(Conv2D(filters = 64, kernel_size = (3,3), strides=(2,2),padding = 'same',input_shape = (28,28,1)))
discriminator.add(LeakyReLU(0.2))

# (14,14,64) --> (7,7,128)

discriminator.add(Conv2D(filters = 128, kernel_size = ((3,3)),strides = (2,2), padding = 'same'))
discriminator.add(LeakyReLU(0.2))

#(7,7,128) --> 6272
discriminator.add(Flatten())
discriminator.add(Dense(100))
discriminator.add(LeakyReLU(0.2))

discriminator.add(Dense(1,activation = 'sigmoid'))
discriminator.compile(loss = keras.losses.binary_crossentropy, optimizer = adam)
discriminator.summary()



Model: "sequential_12"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 14, 14, 64)        640       
                                                                 
 leaky_re_lu_12 (LeakyReLU)  (None, 14, 14, 64)        0         
                                                                 
 conv2d_5 (Conv2D)           (None, 7, 7, 128)         73856     
                                                                 
 leaky_re_lu_13 (LeakyReLU)  (None, 7, 7, 128)         0         
                                                                 
 flatten_1 (Flatten)         (None, 6272)              0         
                                                                 
 dense_8 (Dense)             (None, 100)               627300    
                                                                 
 leaky_re_lu_14 (LeakyReLU)  (None, 100)             

In [30]:
## Combine Model

discriminator.trainable = False
gan_input = Input(shape = (noise_dim,))
x = generator(gan_input)
gan_output = discriminator(x)
gan = Model(inputs = gan_input, outputs = gan_output)
gan.compile(loss = keras.losses.BinaryCrossentropy(), optimizer = adam )
gan.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 sequential_5 (Sequential)   (None, 28, 28, 1)         709057    
                                                                 
 sequential_12 (Sequential)  (None, 1)                 701897    
                                                                 
Total params: 1410954 (5.38 MB)
Trainable params: 708673 (2.70 MB)
Non-trainable params: 702281 (2.68 MB)
_________________________________________________________________


In [32]:
X_train = X_train.reshape(-1,28,28,1)
X_train.shape


(60000, 28, 28, 1)

In [ ]:
## Training Loop

d_losses = []
g_losses = []

for epoch in range(total_epochs):
  epoch_d_loss = 0.0
  epoch_g_loss = 0.0

  # MINI BATCH GRADIENT DESCENT
  for step in range(no_of_batches):
    # Train discriminator
    # get the real data
    idx = np.random.randint(0,6000,half_batch)
    real_img = X_train[idx]
    # get the fake data
    noise = np.random.normal(0,1,(half_batch,noise_dim))
    fake_img = generator.predict(noise)
    # labels
    real_y = np.ones((half_batch,1))
    fake_y = np.zeros((half_batch,1))
    # train discriminator
    d_loss_real = discriminator.train_on_batch(real_img,real_y)
    d_loss_fake = discriminator.train_on_batch(fake_img,fake_y)
    d_loss = 0.5*np.add(d_loss_real,d_loss_fake)
    epoch_d_loss += d_loss[d_losses]
    # Train Generator
    noise = np.random.normal(0,1,(batch_size,noise_dim))
    g_loss = gan.train_on_batch(noise,real_y)
    epoch_g_loss += g_loss[g_losses]

    print(f"Epoch{}")

